# 🩺 Şeker Hastalığı (Diyabet) Teşhisi — Makine Öğrenmesi Projesi

**Proje Özeti:**
- **Problem:** Kişinin sağlık ölçümlerine göre diyabetli olup olmadığını tahmin etmek
- **Problem Türü:** İkili Sınıflandırma (Diyabetli: 1 / Sağlıklı: 0)
- **Veri Seti:** Kaggle — Pima Indians Diabetes Database
- **Algoritmalar:** Lojistik Regresyon, Karar Ağacı, Rastgele Orman

## 📦 Kütüphaneler

Hazır araçları içeri alıyoruz.
- `pandas` → tablo işlemleri (Excel gibi)
- `numpy` → sayısal hesaplamalar
- `matplotlib / seaborn` → grafik çizme
- `sklearn` → makine öğrenmesi modelleri

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split   # veriyi ikiye bölmek için
from sklearn.preprocessing import StandardScaler       # değerleri aynı ölçeğe getirmek için
from sklearn.linear_model import LogisticRegression    # 1. model
from sklearn.tree import DecisionTreeClassifier        # 2. model
from sklearn.ensemble import RandomForestClassifier    # 3. model
from sklearn.metrics import (accuracy_score, precision_score,
                             recall_score, f1_score, confusion_matrix)

sns.set_theme(style="whitegrid", palette="muted")
print("Kütüphaneler başarıyla yüklendi ✅")

## 📁 Bölüm 1 — Veri Seti Tanıtımı

CSV dosyasını okuyup tabloya çeviriyoruz. `df` artık bizim veri tablomuzdur.

| Sütun | Açıklama |
|---|---|
| Pregnancies | Gebelik sayısı |
| Glucose | Kan şekeri değeri |
| BloodPressure | Kan basıncı |
| SkinThickness | Deri kalınlığı |
| Insulin | İnsülin seviyesi |
| BMI | Vücut kitle indeksi |
| DiabetesPedigreeFunction | Aile diyabet geçmişi skoru |
| Age | Yaş |
| **Outcome** | **Hedef: 0 = Sağlıklı, 1 = Diyabetli** |

In [ ]:
# CSV dosyasını oku, df = veri tablomuzdur
df = pd.read_csv("diabetes.csv")

# Kaç satır ve sütun var?
print(f"Satır sayısı  : {df.shape[0]}")
print(f"Sütun sayısı  : {df.shape[1]}")

# İlk 5 satıra göz at
print("\nİlk 5 satır:")
df.head()

In [ ]:
# Kaç kişi sağlıklı (0), kaç kişi diyabetli (1)?
print("Hedef sınıf dağılımı:")
print(df["Outcome"].value_counts())

# Her sütunun veri tipi
print("\nVeri tipleri:")
print(df.dtypes)

## 🔍 Bölüm 2 — Veri Analizi ve Ön İşleme (EDA)

### 2.1 Eksik Veri Temizleme

Kan şekeri, kan basıncı gibi değerler `0` olamaz. Bu sütunlardaki sıfırlar aslında **eksik veriyi** temsil eder. Onları `NaN` (boş) yapıp o sütunun **ortanca değeriyle** dolduruyoruz.

In [ ]:
# 0 olamayacak sütunlar
sifir_olamaz = ["Glucose", "BloodPressure", "SkinThickness", "Insulin", "BMI"]

# Kaç tane sıfır var?
print("Sıfır sayıları:")
print((df[sifir_olamaz] == 0).sum())

# 0 → NaN yap, sonra ortanca ile doldur
df[sifir_olamaz] = df[sifir_olamaz].replace(0, np.nan)
df[sifir_olamaz] = df[sifir_olamaz].fillna(df[sifir_olamaz].median())
print("\nEksik değerler ortanca ile dolduruldu ✅")

### 2.2 Temel İstatistikler

Her sütunun ortalama, min, max gibi özet bilgilerini göster.

In [ ]:
# Ortalama, min, max, standart sapma
df.describe().round(2)

### 2.3 Aykırı Değer Analizi

IQR yöntemiyle çok uç değerleri tespit ediyoruz. Çeyrekler arası aralığın 1.5 katının dışındaki değerler aykırı sayılır.

In [ ]:
# IQR = Q3 - Q1 → bu aralığın dışındakiler aykırı değer
print("Aykırı değer sayıları:")
for col in df.columns[:-1]:
    Q1  = df[col].quantile(0.25)
    Q3  = df[col].quantile(0.75)
    IQR = Q3 - Q1
    aykiri = ((df[col] < Q1 - 1.5*IQR) | (df[col] > Q3 + 1.5*IQR)).sum()
    print(f"  {col:30s}: {aykiri} aykırı değer")

### 2.4 Grafik 1 — Hedef Sınıf Dağılımı

Kaç kişi sağlıklı, kaç kişi diyabetli? Pasta grafikle görselleştir.

In [ ]:
# Pasta grafik: sağlıklı vs diyabetli oranı
fig, ax = plt.subplots(figsize=(6, 5))
df["Outcome"].value_counts().plot.pie(
    labels=["Sağlıklı (0)", "Diyabetli (1)"],
    autopct="%1.1f%%", colors=["#4CAF50", "#F44336"],
    startangle=90, ax=ax
)
ax.set_ylabel("")
ax.set_title("Hedef Sınıf Dağılımı")
plt.tight_layout()
plt.show()

### 2.5 Grafik 2 — Özelliklerin Dağılımı

Her sütunun değer dağılımını histogram ile göster. Verinin nasıl yayıldığını anlamak için kullanılır.

In [ ]:
# Her sütun için histogram
df.hist(bins=20, figsize=(12, 8), color="#5C85D6", edgecolor="white")
plt.suptitle("Özelliklerin Dağılımı", fontsize=14)
plt.tight_layout()
plt.show()

### 2.6 Grafik 3 — Korelasyon Isı Haritası

Hangi özellikler birbirini etkiliyor? Renk ne kadar koyu → ilişki o kadar güçlü.

In [ ]:
# Korelasyon: özellikler arasındaki ilişki
plt.figure(figsize=(9, 7))
sns.heatmap(df.corr(), annot=True, fmt=".2f", cmap="coolwarm", linewidths=0.5)
plt.title("Korelasyon Isı Haritası")
plt.tight_layout()
plt.show()

### 2.7 Grafik 4 — Kan Şekeri ve VKİ İlişkisi

Diyabetliler ile sağlıklıları nokta grafikle karşılaştır. Kan şekeri yükseldikçe diyabet riskinin arttığı görülür.

In [ ]:
# Diyabetli (kırmızı) vs sağlıklı (yeşil) dağılımı
plt.figure(figsize=(8, 5))
sns.scatterplot(data=df, x="Glucose", y="BMI",
                hue="Outcome", palette={0:"#4CAF50", 1:"#F44336"}, alpha=0.7)
plt.title("Kan Şekeri - Vücut Kitle İndeksi İlişkisi")
plt.legend(title="Durum", labels=["Sağlıklı", "Diyabetli"])
plt.tight_layout()
plt.show()

## 🤖 Bölüm 3 — Model Kurma

Veriyi **%80 eğitim / %20 test** olarak ikiye bölüyoruz.
- Eğitim seti → modelin öğrendiği veriler
- Test seti → modelin daha önce görmediği verilerle sınava girdiği kısım

Sınava girmeden önce 80 soruyla çalış, 20 soruyla sınava gir gibi düşün.

3 farklı algoritma deniyoruz:
1. **Lojistik Regresyon** — en basit model, referans noktası
2. **Karar Ağacı** — dal dal karar vererek sınıflandırır
3. **Rastgele Orman** — 100 karar ağacının ortak kararı, en güçlü model

In [ ]:
# X → özellikler (kan şekeri, yaş vs.)
# y → doğru cevaplar (0: sağlıklı, 1: diyabetli)
X = df.drop("Outcome", axis=1)
y = df["Outcome"]

# %80 eğitim / %20 test olarak böl
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Eğitim seti : {X_train.shape[0]} satır")
print(f"Test seti   : {X_test.shape[0]} satır")

# Değerleri aynı ölçeğe getir (Lojistik Regresyon için gerekli)
scaler    = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

In [ ]:
# 3 modeli eğit ve sonuçları kaydet
modeller = {
    "Lojistik Regresyon": (LogisticRegression(max_iter=1000, random_state=42), True),
    "Karar Ağacı"       : (DecisionTreeClassifier(max_depth=5, random_state=42), False),
    "Rastgele Orman"    : (RandomForestClassifier(n_estimators=100, random_state=42), False),
}

sonuclar = {}

for isim, (model, olcekli) in modeller.items():
    # Lojistik Regresyon ölçeklenmiş veri kullanır, diğerleri kullanmaz
    model.fit(X_train_s if olcekli else X_train, y_train)
    y_pred = model.predict(X_test_s if olcekli else X_test)

    # Doğruluk, hassasiyet, duyarlılık ve F1 skorunu hesapla
    sonuclar[isim] = {
        "Doğruluk"   : accuracy_score(y_test, y_pred),
        "Hassasiyet" : precision_score(y_test, y_pred),
        "Duyarlılık" : recall_score(y_test, y_pred),
        "F1 Skoru"   : f1_score(y_test, y_pred),
        "y_pred"     : y_pred,
    }
    print(f"{isim} modeli eğitildi ✅")

## 📊 Bölüm 4 — Model Değerlendirme

### 4.1 Karşılaştırma Tablosu

Hangi metrik ne anlama gelir?
- **Doğruluk** → 100 tahminin kaçı doğru?
- **Hassasiyet** → Diyabetli dediğimin kaçı gerçekten diyabetli?
- **Duyarlılık** → Gerçek diyabetlilerin kaçını yakaladım?
- **F1 Skoru** → İkisinin dengeli ortalaması

In [ ]:
# Tüm modellerin skorlarını tablo olarak göster
tablo = pd.DataFrame(
    {k: {m: round(v, 4) for m, v in sonuclar[k].items() if m != "y_pred"}
     for k in sonuclar}
).T

print("MODEL KARŞILAŞTIRMASI")
print("=" * 55)
tablo

In [ ]:
# En yüksek F1 skoruna sahip modeli bul ve göster
en_iyi = max(sonuclar, key=lambda k: sonuclar[k]["F1 Skoru"])
print(f"✅ En iyi model  : {en_iyi}")
print(f"   Doğruluk oranı: %{sonuclar[en_iyi]['Doğruluk']*100:.2f}")

### 4.2 Karışıklık Matrisi

Modelin neyi doğru, neyi yanlış tahmin ettiğini gösterir.
- **Köşegen** → doğru tahminler ✅
- **Köşegen dışı** → yanlış tahminler ❌

In [ ]:
# Karışıklık matrisi: doğru ve yanlış tahminleri göster
cm = confusion_matrix(y_test, sonuclar[en_iyi]["y_pred"])
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["Sağlıklı", "Diyabetli"],
            yticklabels=["Sağlıklı", "Diyabetli"])
plt.title(f"Karışıklık Matrisi — {en_iyi}")
plt.xlabel("Tahmin Edilen")
plt.ylabel("Gerçek Değer")
plt.tight_layout()
plt.show()

### 4.3 Model Performans Karşılaştırma Grafiği

Tüm modellerin skorlarını yan yana çubuk grafik olarak göster.

In [ ]:
# Çubuk grafik: tüm modellerin tüm metrikleri
tablo[["Doğruluk","Hassasiyet","Duyarlılık","F1 Skoru"]].plot(
    kind="bar", figsize=(9,5), edgecolor="white"
)
plt.title("Model Performans Karşılaştırması")
plt.ylabel("Skor")
plt.ylim(0, 1)
plt.xticks(rotation=20)
plt.legend(loc="lower right")
plt.tight_layout()
plt.show()

### 4.4 Özellik Önem Sıralaması

Rastgele Orman modeline göre hangi özellik diyabet tahmininde **en belirleyici**? Çubuk ne kadar uzun → o özellik o kadar önemli.

In [ ]:
# Rastgele Orman'ın her özelliğe verdiği önem skorunu göster
rf_model = modeller["Rastgele Orman"][0]
onem = pd.Series(rf_model.feature_importances_, index=X.columns).sort_values()

plt.figure(figsize=(7, 5))
onem.plot(kind="barh", color="#5C85D6")
plt.title("Rastgele Orman — Özellik Önem Sıralaması")
plt.xlabel("Önem Skoru")
plt.tight_layout()
plt.show()

## ✅ Sonuç

Bu projede şeker hastalığı teşhisi için üç farklı makine öğrenmesi algoritması uygulanmış ve karşılaştırılmıştır.

- **En başarılı model:** Rastgele Orman
- **En belirleyici özellik:** Kan şekeri (Glucose)
- **Önemli bulgu:** Veri setinde sınıf dengesizliği mevcuttur (%71 sağlıklı, %29 diyabetli). Bu nedenle F1 skoru, doğruluk oranına ek olarak değerlendirmeye alınmıştır.